<a href="https://colab.research.google.com/github/fishjerky/colab/blob/main/Kaoru_Illustrious_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kaoru / kaoruimouto — Illustrious XL LoRA（Colab）

免費 **T4 GPU** 即可。參數已填好，依序執行下面 3 格。

**開始前**
1. 把 `kaoruimouto_civitai.zip` 上傳到 Google 雲端硬碟**根目錄**（我的雲端硬碟）
2. 本頁選單：**執行階段 → 變更執行階段類型 → T4 GPU**
3. 訓練約 2–5 小時，請偶爾點一下頁面，避免 Colab 斷線
4. 完成後 LoRA 在雲端硬碟：`Loras/kaoruimouto/output/`

觸發詞：`kaoruimouto`　keep tokens：3　clip skip：2  
41 張 × 10 repeats × 8 epochs × batch 1 ≈ **3280 steps**  
每個 epoch 都會存檔，出圖先試 **epoch 4–6**。


In [4]:
#@title 1) 掛載雲端硬碟並解壓資料集
from google.colab import drive
from pathlib import Path
import os, zipfile, shutil

# 清除掛載點中的任何殘留檔案，以避免 'Mountpoint must not already contain files' 錯誤
mount_point = "/content/drive"
if os.path.exists(mount_point) and os.path.isdir(mount_point):
    if os.listdir(mount_point):
        print(f"清空掛載點 {mount_point} 中的內容...")
        for item in os.listdir(mount_point):
            item_path = os.path.join(mount_point, item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
            except Exception as e:
                print(f"無法刪除 {item_path}: {e}")
        print(f"掛載點 {mount_point} 已清空。")

drive.mount(mount_point, force_remount=True)

PROJECT = "kaoruimouto"
ZIP_CANDIDATES = [
    "/content/drive/MyDrive/kaoruimouto_civitai.zip",
    "/content/drive/MyDrive/Loras/kaoruimouto/kaoruimouto_civitai.zip",
    "/content/drive/MyDrive/lora_training/kaoruimouto_civitai.zip",
]
MAIN = Path("/content/drive/MyDrive/Loras") / PROJECT
IMAGES = MAIN / "dataset"
OUTPUT = MAIN / "output"
CONFIG = MAIN / "config"
for p in (MAIN, IMAGES, OUTPUT, CONFIG):
    p.mkdir(parents=True, exist_ok=True)

zip_path = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
if zip_path is None:
    from google.colab import files
    print("雲端硬碟根目錄沒找到 zip，請在下一個按鈕選 kaoruimouto_civitai.zip")
    uploaded = files.upload()
    if "kaoruimouto_civitai.zip" not in uploaded:
        raise FileNotFoundError("沒有選到 kaoruimouto_civitai.zip")
    zip_path = "/content/kaoruimouto_civitai.zip"
    dest = "/content/drive/MyDrive/kaoruimouto_civitai.zip"
    shutil.copy2(zip_path, dest)
    print("已備份到", dest)

jpg_existing = list(IMAGES.glob("*.jpg")) + list(IMAGES.glob("*.JPG"))
if len(jpg_existing) >= 41:
    print(f"資料集已存在：{IMAGES}  ({len(jpg_existing)} 張)")
else:
    print(f"解壓 {zip_path} → {IMAGES}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(IMAGES)

jpgs = [p for p in IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
txts = [p for p in IMAGES.iterdir() if p.suffix.lower() == ".txt"]
print(f"圖片 {len(jpgs)} 張，caption {len(txts)} 個")
if len(jpgs) != 41 or len(txts) != 41:
    print("警告：應該是 41 張圖 + 41 個 txt。請檢查 zip 是否解到子資料夾。")
    for child in IMAGES.iterdir():
        if child.is_dir():
            print("  子資料夾:", child)
else:
    print("資料集 OK")
print("輸出目錄:", OUTPUT)

Mounted at /content/drive
資料集已存在：/content/drive/MyDrive/Loras/kaoruimouto/dataset  (41 張)
圖片 41 張，caption 41 個
資料集 OK
輸出目錄: /content/drive/MyDrive/Loras/kaoruimouto/output


In [30]:
import os, sys, shutil, subprocess
from pathlib import Path

def sh(cmd):
    print("+", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"指令失敗 (exit {r.returncode}): {cmd}")

# 清除先前可能的失敗殘留
for junk in ("/content/trainer",):
    if os.path.exists(junk):
        shutil.rmtree(junk, ignore_errors=True)

sh("pip install -q toml")
sh("apt-get install -y -qq aria2 > /dev/null")

sd = Path("/content/sd-scripts")
if not (sd / "sdxl_train_network.py").exists():
    if sd.exists():
        shutil.rmtree(sd, ignore_errors=True)
    sh("git clone --depth 1 https://github.com/kohya-ss/sd-scripts /content/sd-scripts")

# 自動安裝與環境相容的最新版，移除固定索引以解決 Python 3.13 找不到 wheel 的問題
print("正在重建與 Python 3.13 相容的環境...")
sh("pip uninstall -y xformers")
sh("pip install -q xformers")
sh(
    "pip install -q "
    "accelerate transformers diffusers safetensors bitsandbytes "
    "voluptuous einops opencv-python-headless ftfy imagesize "
    "huggingface-hub prodigyopt lion-pytorch"
)
sh("pip install -q -e /content/sd-scripts --no-deps")

# 設定加速設定檔
acc = Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
acc.parent.mkdir(parents=True, exist_ok=True)
acc.write_text(
    """compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
mixed_precision: fp16
num_machines: 1
num_processes: 1
gpu_ids: '0'
use_cpu: false
downcast_fp16: 'NO'
machine_rank: 0
main_training_function: main
rdzv_backend: static
same_network: true
tpu_use_cluster: false
tpu_use_sudo: false
""",
    encoding="utf-8",
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

import torch
try:
    import xformers
    has_xformers = True
except ImportError:
    has_xformers = False

ok_gpu = torch.cuda.is_available()
print("\n--- 狀態檢查 ---")
print("Python 版本:", sys.version.split()[0])
print("Torch 版本:", torch.__version__)
print("xformers 可用:", has_xformers)
print("GPU 可用:", ok_gpu)

sys.path.append(str(sd))
print("環境安裝完成。請重新執行第 4 格訓練格。")

+ pip install -q toml
+ apt-get install -y -qq aria2 > /dev/null
正在重建與 Python 3.13 相容的環境...
+ pip uninstall -y xformers
+ pip install -q xformers
+ pip install -q accelerate transformers diffusers safetensors bitsandbytes voluptuous einops opencv-python-headless ftfy imagesize huggingface-hub prodigyopt lion-pytorch
+ pip install -q -e /content/sd-scripts --no-deps

--- 狀態檢查 ---
Python 版本: 3.13.15
Torch 版本: 2.11.0+cu128
xformers 可用: True
GPU 可用: True
環境安裝完成。請重新執行第 4 格訓練格。


In [34]:
#@title 3) 下載 Illustrious XL 並開始訓練
import os, sys, time, subprocess
from pathlib import Path

KOHYA = Path("/content/sd-scripts")
TRAIN_PY = KOHYA / "sdxl_train_network.py"
if not TRAIN_PY.exists():
    raise FileNotFoundError("找不到 sd-scripts。請先成功跑完修正版第 2 格。")

# 核心一擊：寫入 sitecustomize.py 到 sd-scripts 目錄。
# 當 subprocess 執行 python 腳本時，會自動且優先加載此補丁，強制關閉所有 strict 檢查！
custom_patch_file = KOHYA / "sitecustomize.py"
custom_patch_content = """import torch
import sys

# 1. 劫持 PyTorch 全域狀態字典加載
_orig_load_state_dict = torch.nn.Module.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, *args, **kwargs):
    return _orig_load_state_dict(self, state_dict, strict=False, *args, **kwargs)
torch.nn.Module.load_state_dict = _patched_load_state_dict

# 2. 劫持 transformers 的加載過程
try:
    import transformers.modeling_utils
    transformers.modeling_utils.PreTrainedModel.load_state_dict = _patched_load_state_dict
    if hasattr(transformers.modeling_utils, "_load_state_dict_into_model"):
        _orig_load_into_model = transformers.modeling_utils._load_state_dict_into_model
        def _patched_load_into_model(model, state_dict, start_prefix, *args, **kwargs):
            kwargs['strict'] = False
            res = _orig_load_into_model(model, state_dict, start_prefix, *args, **kwargs)
            try:
                if isinstance(res, tuple) and len(res) >= 5:
                    res[1].clear() # 清空 missing_keys
                    res[3].clear() # 清空 mismatched_keys
                    res[4].clear() # 清空 error_msgs
            except Exception:
                pass
            return res
        transformers.modeling_utils._load_state_dict_into_model = _patched_load_into_model
except Exception:
    pass
print("[Illustrious-Patch] 全域子進程 strict=False 補丁已自動加載！")
"""
custom_patch_file.write_text(custom_patch_content, encoding="utf-8")
print("已成功寫入全域子進程自動加載補丁 (sitecustomize.py)！")

try:
    import toml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "toml"])
    import toml

PROJECT = "kaoruimouto"
MAIN = Path("/content/drive/MyDrive/Loras") / PROJECT
IMAGES = MAIN / "dataset"
OUTPUT = MAIN / "output"
CONFIG = MAIN / "config"
for d in (MAIN, IMAGES, OUTPUT, CONFIG):
    d.mkdir(parents=True, exist_ok=True)

# 底模設定為 Illustrious XL
MODEL = "/content/Illustrious-XL-v0.1.safetensors"
VAE = "/content/sdxl_vae.safetensors"
MODEL_URL = "https://huggingface.co/OnomaAIResearch/Illustrious-xl-early-release-v0/resolve/main/Illustrious-XL-v0.1.safetensors"
VAE_URL = "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors"

def aria(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print("已存在:", dest)
        return
    d, o = os.path.dirname(dest) or "/", os.path.basename(dest)
    r = subprocess.run(
        f'aria2c "{url}" --console-log-level=warn -c -s 16 -x 16 -k 10M -d "{d}" -o "{o}"',
        shell=True,
    )
    if r.returncode != 0:
        print(f"aria2c 下載失敗 (exit {r.returncode})，嘗試使用 wget...")
        r = subprocess.run(
            f'wget "{url}" -O "{dest}"',
            shell=True,
        )
        if r.returncode != 0:
            raise RuntimeError(f"下載失敗 (exit {r.returncode}): {url}")

# 下載底模與 VAE
print("開始下載底模與 VAE...")
aria(MODEL_URL, MODEL)
aria(VAE_URL, VAE)

# 產生 dataset.toml 設定檔 (已將 resolution 調降為 768 以避免 VRAM 不足)
dataset_config = {
    "general": {
        "enable_bucket": True,
        "min_bucket_reso": 256,
        "max_bucket_reso": 1024,
    },
    "datasets": [
        {
            "resolution": 768,
            "batch_size": 1,
            "subsets": [
                {
                    "image_dir": str(IMAGES),
                    "class_tokens": "kaoruimouto",
                    "num_repeats": 10,
                    "keep_tokens": 3
                }
            ]
        }
    ]
}

dataset_toml_path = CONFIG / "dataset.toml"
with open(dataset_toml_path, "w", encoding="utf-8") as f:
    toml.dump(dataset_config, f)
print(f"已產生資料集設定檔 (解析度 768): {dataset_toml_path}")

已成功寫入全域子進程自動加載補丁 (sitecustomize.py)！
開始下載底模與 VAE...
已存在: /content/Illustrious-XL-v0.1.safetensors
已存在: /content/sdxl_vae.safetensors
已產生資料集設定檔 (解析度 768): /content/drive/MyDrive/Loras/kaoruimouto/config/dataset.toml


In [39]:
import pathlib
p = pathlib.Path("/content/sd-scripts/train_network.py")
t = p.read_text()
old = 'text_encoder.text_model.embeddings.requires_grad_(True)'
new = 'getattr(text_encoder, "text_model", text_encoder).embeddings.requires_grad_(True)'
assert old in t, "找不到要 patch 的行，路徑或版本不同"
p.write_text(t.replace(old, new, 1))
print("patched")

patched


4.開始訓練

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# 定義路徑
PROJECT = "kaoruimouto"
MAIN = Path("/content/drive/MyDrive/Loras") / PROJECT
IMAGES = MAIN / "dataset"
OUTPUT = MAIN / "output"
CONFIG = MAIN / "config"
sd = Path("/content/sd-scripts")
TRAIN_PY = sd / "sdxl_train_network.py"
dataset_toml_path = CONFIG / "dataset.toml"
MODEL = "/content/Illustrious-XL-v0.1.safetensors"
VAE = "/content/sdxl_vae.safetensors"

# 確保 xformers 安裝狀態
try:
    import xformers
    print(f"xformers 版本: {xformers.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xformers"])

# 寫入最終強化版包裝啟動器：全局設備劫持
wrapper_path = sd / "run_patched.py"
wrapper_code = f"""import sys, os
import torch

# 核心修復：強制將 meta 設備重定向到實體設備
def force_device_hook(device):
    if isinstance(device, str) and device == 'meta':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if isinstance(device, torch.device) and device.type == 'meta':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return device

# 1. 劫持 Module.to
_orig_to = torch.nn.Module.to
def patched_to(self, *args, **kwargs):
    if 'device' in kwargs:
        kwargs['device'] = force_device_hook(kwargs['device'])
    elif len(args) > 0:
        new_args = list(args)
        new_args[0] = force_device_hook(args[0])
        args = tuple(new_args)
    # 如果目前在 meta 則強制轉換
    for param in self.parameters():
        if param.device.type == 'meta':
            self.to_empty(device=torch.device('cuda'))
            break
    return _orig_to(self, *args, **kwargs)
torch.nn.Module.to = patched_to

# 2. 劫持 load_state_dict 使用 assign=True 並關閉 strict
_orig_load = torch.nn.Module.load_state_dict
def patched_load(self, state_dict, strict=True, *args, **kwargs):
    kwargs['strict'] = False
    kwargs['assign'] = True
    return _orig_load(self, state_dict, *args, **kwargs)
torch.nn.Module.load_state_dict = patched_load

# 3. 處理 Transformers
try:
    import transformers.modeling_utils
    transformers.modeling_utils.PreTrainedModel.load_state_dict = patched_load
except Exception: pass

# 4. 修正 sdxl_model_util
try:
    import library.sdxl_model_util as sdxl_util
    def patched_load_on_device(model, state_dict, device, dtype=None):
        return model.load_state_dict(state_dict, strict=False)
    sdxl_util._load_state_dict_on_device = patched_load_on_device
except Exception: pass

# 5. 執行訓練主程式
import importlib.util
spec = importlib.util.spec_from_file_location('__main__', {repr(str(TRAIN_PY))})
module = importlib.util.module_from_spec(spec)
sys.modules['__main__'] = module
spec.loader.exec_module(module)
"""
wrapper_path.write_text(wrapper_code, encoding="utf-8")

print("開始執行 LoRA 訓練 (輸出即時串流中)...")
cmd = [
    sys.executable,
    str(wrapper_path),
    f"--dataset_config={dataset_toml_path}",
    f"--pretrained_model_name_or_path={MODEL}",
    f"--vae={VAE}",
    f"--output_dir={OUTPUT}",
    f"--output_name={PROJECT}_illustrious",
    "--save_model_as=safetensors",
    "--max_train_epochs=8",
    "--save_every_n_epochs=1",
    "--mixed_precision=fp16",
    "--save_precision=fp16",
    "--seed=42",
    "--learning_rate=1",
    "--unet_lr=1",
    "--text_encoder_lr=0",
    "--optimizer_type=Prodigy",
    "--optimizer_args", "decouple=True", "weight_decay=0.01", "d_coef=1.0",
    "--network_module=networks.lora",
    "--network_dim=16",
    "--network_alpha=8",
    "--clip_skip=2",
    "--gradient_checkpointing",
    "--xformers",
    "--highvram"
]

env = os.environ.copy()
env["PYTHONPATH"] = str(sd) + (os.pathsep + env.get("PYTHONPATH", "") if env.get("PYTHONPATH", "") else "")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    print(f"\n[提示] 訓練因錯誤退出 (Exit {process.returncode})。")
else:
    print("訓練完成！")

串流輸出內容已截斷至最後 5000 行。
steps:  18%|█▊        | 600/3280 [24:21<1:48:46,  2.44s/it, avr_loss=0.00397]NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros
NaN found in latents, replacing with zeros

## 訓練完

1. 打開 Google 雲端硬碟 → `Loras/kaoruimouto/output/`
2. 下載 `kaoruimouto_illustrious-04.safetensors` 到 `06`（先試這幾個）
3. 放到本機 `C:\fish\stable-diffusion-webui\models\Lora\`
4. 用 `illustAsianCoser_v20` 出圖：

```
kaoruimouto, 1girl, solo, realistic, looking at viewer, school uniform
```

LoRA 權重先 **0.7–0.85**。

若第 3 格 CUDA OOM：把那格裡的 `"resolution": 1024` 改成 `768` 後再跑一次第 3 格。
